In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import style
from io import StringIO
import pandas as pd
import matplotlib.pyplot as plt
import time

**PLOT TXT**

In [ ]:
df = pd.read_csv(r"C:\Users\vinic\Desktop\ESPs3\carro ligado acelerando até 2k rpm depois intertimente.txt", sep=r"\s+", header=None)

In [ ]:
df.columns

In [ ]:
df_log_cleaned = df.drop(columns=[1])
df_log_cleaned.columns = ['timestamp'] + ['can_id'] + [f'data_{i}' for i in range(0, 8)]
df_log_cleaned.head()

In [ ]:
for col in [f'data_{i}' for i in range(0, 8)]:
    df_log_cleaned[col] = df_log_cleaned[col].apply(lambda x: int(x, 16) if isinstance(x, str) and x.startswith("0x") else 0)

In [ ]:
df_log_cleaned.head()

In [ ]:
df_log_cleaned['can_id'].unique()

In [ ]:
for can_id in df_log_cleaned['can_id'].unique():
    
    df_selected = df_log_cleaned[df_log_cleaned['can_id'] == can_id] 
    
    for i in range(8):  # data_0 to data_7
        fig = plt.figure(figsize=(20, 8))
        plt.plot(df_selected['timestamp'], df_selected[f'data_{i}'], label=f'data_{i}')
        
        plt.title(f'Data bytes over time for CAN ID {can_id}, data_{i}')
        plt.xlabel('Timestamp')
        plt.ylabel('Data Byte Value')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

**PLOT CSV**

In [ ]:
df_quente = pd.read_csv(r"C:\Users\vinic\Desktop\ESPs3\can_log_carro_quente.csv")

In [ ]:
df_quente.head()

In [ ]:
df_quente['data'] = df_quente['data'].fillna('')

In [ ]:
max_len = 8 
def split_and_pad(row):
    parts = row.strip().split()
    ints = [int(p) for p in parts]
    return ints + [0] * (max_len - len(ints))

data_expanded = df_quente['data'].apply(split_and_pad).apply(pd.Series)
data_expanded.columns = [f'data_{i}' for i in range(max_len)]
df_quente = pd.concat([df_quente.drop(columns=['data']), data_expanded], axis=1)

In [ ]:
df_quente['timestamp'] = pd.to_datetime(df_quente['timestamp'])

In [ ]:
for can_id, group in df_quente.groupby('can_id'):
    for i in range(8):  # data_0 to data_7
    #     if f'data_{i}' in group.columns:
        plt.figure(figsize=(20, 5))
        plt.plot(group['timestamp'], group[f'data_{i}'], label=f'data_{i}')
        
        plt.title(f'Data bytes over time for CAN ID {can_id}')
        plt.xlabel('Timestamp')
        plt.ylabel('Data Byte Value')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()